In [159]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression


In [161]:
df = pd.read_csv("heart (1).csv")

In [163]:
df

,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall,output
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0


In [165]:
# data cleaning
df.isnull().sum()

age         0
sex         0
cp          0
trtbps      0
chol        0
fbs         0
restecg     0
thalachh    0
exng        0
oldpeak     0
slp         0
caa         0
thall       0
output      0
dtype: int64

In [167]:
df.duplicated().sum()

1

In [169]:
df.drop_duplicates(inplace=True)

In [171]:
df.duplicated().sum()

0

In [173]:
df.isnull().sum()

age         0
sex         0
cp          0
trtbps      0
chol        0
fbs         0
restecg     0
thalachh    0
exng        0
oldpeak     0
slp         0
caa         0
thall       0
output      0
dtype: int64

In [175]:
# data integration

subset1= df[['age','sex','chol']]
subset2 = df[['oldpeak','slp','caa']]

In [177]:
merged_df = pd.concat([subset1,subset2],axis =1)
merged_df

,age,sex,chol,oldpeak,slp,caa
0,63,1,233,2.3,0,0
1,37,1,250,3.5,0,0
2,41,0,204,1.4,2,0
3,56,1,236,0.8,2,0
4,57,0,354,0.6,2,0
...,...,...,...,...,...,...
298,57,0,241,0.2,1,0
299,45,1,264,1.2,1,0
300,68,1,193,3.4,1,2
301,57,1,131,1.2,1,1


In [180]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 302 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       302 non-null    int64  
 1   sex       302 non-null    int64  
 2   cp        302 non-null    int64  
 3   trtbps    302 non-null    int64  
 4   chol      302 non-null    int64  
 5   fbs       302 non-null    int64  
 6   restecg   302 non-null    int64  
 7   thalachh  302 non-null    int64  
 8   exng      302 non-null    int64  
 9   oldpeak   302 non-null    float64
 10  slp       302 non-null    int64  
 11  caa       302 non-null    int64  
 12  thall     302 non-null    int64  
 13  output    302 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 35.4 KB


In [182]:
def remove_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    threshold = 1.5 * IQR
    outlier_mask = (column < Q1 - threshold)| (column > Q3 + threshold)
    return column[~outlier_mask]

In [187]:
#remove outliers for each column using a loop 
col_name = ['cp', 'thalachh','exng', 'oldpeak', 'slp', 'caa']
for col in col_name:
    df[col] = remove_outliers(df[col])

In [189]:
scaler = StandardScaler()


In [191]:
df.isna().sum()

age          0
sex          0
cp           0
trtbps       0
chol         0
fbs          0
restecg      0
thalachh     1
exng         0
oldpeak      5
slp          0
caa         24
thall        0
output       0
dtype: int64

In [193]:
df.dropna(inplace= True)

In [195]:
X = df[['cp', 'thalachh','exng', 'oldpeak', 'slp', 'caa']]
y = df.output

In [198]:
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state = 0)

x_train.shape,x_test.shape,y_train.shape,y_test.shape

((206, 6), (69, 6), (206,), (69,))

In [200]:
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

In [202]:
y_train = np.array(y_train).reshape(-1,1)
y_test = np.array(y_test).reshape(-1,1)


In [204]:
y_train.shape

(206, 1)

In [206]:
y_test.shape

(69, 1)

In [208]:
from sklearn.linear_model import LogisticRegression

In [210]:
model = LogisticRegression()

In [212]:
model.fit(x_train_scaled,y_train)

C:\Users\shubh\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression()

In [214]:
y_pred = model.predict(x_test_scaled)

In [222]:
from sklearn.metrics import accuracy_score,confusion_matrix
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.7681159420289855


In [224]:
from sklearn.tree import DecisionTreeClassifier
tc=DecisionTreeClassifier(criterion='entropy')
tc.fit(x_train_scaled,y_train)
y_pred=tc.predict(x_test_scaled)

print("Training Accuracy Score :",accuracy_score(y_pred,y_test))
print("Training Confusion Matrix  :",confusion_matrix(y_pred,y_test))

Training Accuracy Score : 0.7536231884057971
Training Confusion Matrix  : [[21  6]
 [11 31]]
